# Test of G5 (`05_release_activity_shift`)

This notebook tests `05_release_activity_shift` with small hand-made tables in the schema `github_lakehouse.test_gold_g5`. It runs the real notebook with `%run` and `test_schema = test_gold_g5`, so **production tables are never written**. The last check section proves that.

It has the same sections as the tests of G1, G2, G3, G4 and G6. Three things are different, because G5 is different:

- The hand-made rows are releases and the activity around them, and a small source-hour manifest.
- G5 is rebuilt completely on every run. So a run for one source date also rebuilds everything, and a source date without rows is not `NO_INPUT`.
- The test also changes the manifest, to prove that a release is left out when one hour of its windows is not complete.

**Before you run it:** `08_gold_audit_table` and `07_source_hour_manifest` must have run once, and the Silver tables must exist. The test copies the definitions of the tables. It does not read their rows.

## How to read the output

- Every check section prints a table with one line per check: `PASS` or `FAIL`, the actual value and the expected value.
- A section stops with an error when one of its checks failed.
- **Four cells are EXPECTED to fail.** The text before each of them says so. "Run all" stops at such a cell. Then click the cell **after** it and choose "Run all below". It does no harm if one of these four cells runs more than once.
- **The result of the whole test is the cell "Result of the whole test" near the end.** It prints `TEST PASSED` only when all 13 sections ran and every check passed. If a section failed or was left out, it stops with `TEST FAILED` and names the section.
- The last cell drops the test schema.

## What is tested

| Test of the contract | Where |
|---|---|
| No input | T3, T4: empty Silver tables give `NO_INPUT` and leave Gold empty |
| Schema | T7: the Gold table has exactly the columns and types of the contract |
| The seven anchor rules | T7: 29 release events, at least one for every reason. An event that breaks several rules is counted once, under the first one. A draft never takes the place of a valid event of the same release. A release is identified by repository and release ID together |
| The windows | T7: events exactly at the start, exactly at r, in the last second and exactly at the end of a window. A window uses the published time, not the event time. 48 or 49 hours, at the edges of the week and around an hour that is not complete |
| Exact values | T7: every count of every row, worked out by hand. Two releases of one repository whose windows overlap |
| Only the five tables | T7: an issue comment and a review in the window are not counted. Events of another repository are not counted |
| Missing values | T7: a push without a commit count, a push with 0 commits, an activity event without an event time |
| Zero base | T7: a row without events before has the percent change NULL and the baseline flag true, also when it has events after |
| Rerun | T8, T9: a second run, now with a source date, changes nothing |
| Late event | T10, T11: a late fork changes two rows, a late release adds a row, and a late release event that is earlier takes the anchor from the event that had it |
| Removed event | T12, T13: removed rows lower a row, remove a row and give a release another anchor |
| A source date without rows | T14, T15: still a complete rebuild, `SUCCEEDED`, not `NO_INPUT` |
| Coverage | T14, T15: an hour that is not complete removes the release whose window touches it. T19, T20: when the hour is complete again, the release is back |
| Failed check | T16 to T18: two failing quality gates leave Gold unchanged, then a recovery run |
| Bad parameter | T21 to T23: an invalid `source_date`, `test_schema` or `run_id` stops the notebook in its parameter step. T24: nothing was written |
| Production safety | T24: no production table changed, the manifest included |

## T1. Setup: test schema and empty tables

The test schema is dropped and created again, so every test run starts clean.

`CREATE TABLE ... LIKE` makes an empty table with the same columns as another table. The test tables are empty copies of the real tables, with the same names, only in the test schema:

- the six Silver tables the notebook reads,
- two Silver tables it does **not** read, issue comments and reviews, to prove that they are not counted,
- the source-hour manifest,
- the audit table.

The Gold test table is created by the notebook itself, in its first run.

In [0]:
TEST_SCHEMA = "test_gold_g5"

TEST_SILVER = f"github_lakehouse.{TEST_SCHEMA}"
PROD_SILVER = "github_lakehouse.silver"

# The six Silver tables that the notebook reads, and two it does not read.
SILVER_TABLE_NAMES = [
    "release_events",
    "push_events",
    "pull_request_events",
    "issue_events",
    "watch_events",
    "fork_events",
    "issue_comment_events",
    "pull_request_review_events",
]

# The Silver test tables.
TEST_RELEASE = TEST_SILVER + ".release_events"
TEST_PUSH = TEST_SILVER + ".push_events"
TEST_PULL_REQUEST = TEST_SILVER + ".pull_request_events"
TEST_ISSUE = TEST_SILVER + ".issue_events"
TEST_WATCH = TEST_SILVER + ".watch_events"
TEST_FORK = TEST_SILVER + ".fork_events"
TEST_ISSUE_COMMENT = TEST_SILVER + ".issue_comment_events"
TEST_REVIEW = TEST_SILVER + ".pull_request_review_events"

TEST_MANIFEST = f"github_lakehouse.{TEST_SCHEMA}.source_hour_manifest"
TEST_GOLD = f"github_lakehouse.{TEST_SCHEMA}.release_activity_shift"
TEST_STAGING = f"github_lakehouse.{TEST_SCHEMA}.release_activity_shift_staging"
TEST_AUDIT = f"github_lakehouse.{TEST_SCHEMA}.gold_pipeline_runs"
TEST_COPY = f"github_lakehouse.{TEST_SCHEMA}.gold_rows_after_run_1"

PROD_AUDIT = "github_lakehouse.audit.gold_pipeline_runs"
PROD_MANIFEST = "github_lakehouse.audit.source_hour_manifest"
PROD_GOLD = "github_lakehouse.gold.release_activity_shift"
PROD_STAGING = "github_lakehouse.gold_staging.release_activity_shift"

# Only the columns that G5 reads are filled. The other columns of the Silver tables stay NULL.
RELEASE_COLUMNS = (
    "(event_id, event_timestamp, event_date, repo_id, repo_name, "
    "release_action, release_id, release_tag, release_published_at, release_draft, release_prerelease)"
)
PUSH_COLUMNS = "(event_id, event_timestamp, event_date, repo_id, repo_name, push_commit_count)"
ACTIVITY_COLUMNS = "(event_id, event_timestamp, event_date, repo_id, repo_name)"

# Every time in this test is UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Every check section writes its number of failed checks into this dictionary.
# The cell "Result of the whole test" reads it at the end.
section_results = {}

if not spark.catalog.tableExists(PROD_AUDIT):
    raise RuntimeError(f"{PROD_AUDIT} does not exist. Run 08_gold_audit_table first.")

if not spark.catalog.tableExists(PROD_MANIFEST):
    raise RuntimeError(f"{PROD_MANIFEST} does not exist. Run 07_source_hour_manifest first.")

for table_name in SILVER_TABLE_NAMES:
    if not spark.catalog.tableExists(PROD_SILVER + "." + table_name):
        raise RuntimeError(
            f"{PROD_SILVER}.{table_name} does not exist. Run the Silver notebook of this table first."
        )

# Only a schema whose name starts with test_ is ever dropped.
if not TEST_SCHEMA.startswith("test_"):
    raise RuntimeError("Refusing to drop a schema that is not a test schema.")

spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")
spark.sql(f"CREATE SCHEMA github_lakehouse.{TEST_SCHEMA}")

for table_name in SILVER_TABLE_NAMES:
    spark.sql(f"CREATE TABLE {TEST_SILVER}.{table_name} LIKE {PROD_SILVER}.{table_name}")

spark.sql(f"CREATE TABLE {TEST_MANIFEST} LIKE {PROD_MANIFEST}")
spark.sql(f"CREATE TABLE {TEST_AUDIT} LIKE {PROD_AUDIT}")

print("Test schema ready:", TEST_SCHEMA)

## T2. Remember the state of the production tables

The Delta version of a table goes up with every write. The versions are read now and again at the end. If they are the same, the test did not write to production.

The manifest is read too. The notebook only reads it, so its version must not change either.

The production Gold table and its staging copy may not exist yet. The value `-1` means: the table does not exist.

In [0]:
prod_audit_version_before = spark.sql(
    f"DESCRIBE HISTORY {PROD_AUDIT} LIMIT 1"
).collect()[0]["version"]

prod_manifest_version_before = spark.sql(
    f"DESCRIBE HISTORY {PROD_MANIFEST} LIMIT 1"
).collect()[0]["version"]

if spark.catalog.tableExists(PROD_GOLD):
    prod_gold_version_before = spark.sql(
        f"DESCRIBE HISTORY {PROD_GOLD} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_gold_version_before = -1

if spark.catalog.tableExists(PROD_STAGING):
    prod_staging_version_before = spark.sql(
        f"DESCRIBE HISTORY {PROD_STAGING} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_staging_version_before = -1

print("Production audit version   :", prod_audit_version_before)
print("Production manifest version:", prod_manifest_version_before)
print("Production Gold version    :", prod_gold_version_before)
print("Production staging version :", prod_staging_version_before)

### A three-line summary of every audit row

An audit row holds three maps: the Silver rows read per table, the rows left out, and the quality numbers. The checks below compare them as text. This view writes them into three readable lines, for example:

`releases=29 push=12 pull_requests=4 issues=4 watches=6 forks=4`

`not_published=2 draft=3 no_published_at=1 prerelease_unknown=1 no_release_id=1 repeated=2 not_covered=8`

`release_events=29 anchors=11 prerelease=2 baseline_zero=6 without_activity=5 manifest_hours=168 complete_hours=167`

The second line holds the seven reasons, in the order of the rules, with short names. The view is not a copy. It always reads the audit table as it is at that moment.

In [0]:
audit_summary = spark.sql(f"""
    SELECT
        run_id,
        concat(
            'releases=',         CAST(source_rows_read['release_events'] AS STRING),
            ' push=',            CAST(source_rows_read['push_events'] AS STRING),
            ' pull_requests=',   CAST(source_rows_read['pull_request_events'] AS STRING),
            ' issues=',          CAST(source_rows_read['issue_events'] AS STRING),
            ' watches=',         CAST(source_rows_read['watch_events'] AS STRING),
            ' forks=',           CAST(source_rows_read['fork_events'] AS STRING)
        ) AS source_summary,
        concat(
            'not_published=',        CAST(quality_metrics['NOT_PUBLISHED'] AS STRING),
            ' draft=',               CAST(quality_metrics['DRAFT_OR_UNKNOWN'] AS STRING),
            ' no_published_at=',     CAST(quality_metrics['MISSING_PUBLISHED_AT'] AS STRING),
            ' prerelease_unknown=',  CAST(quality_metrics['PRERELEASE_UNKNOWN'] AS STRING),
            ' no_release_id=',       CAST(quality_metrics['MISSING_RELEASE_ID'] AS STRING),
            ' repeated=',            CAST(quality_metrics['REPEATED_RELEASE_EVENT'] AS STRING),
            ' not_covered=',         CAST(quality_metrics['WINDOW_NOT_COVERED'] AS STRING)
        ) AS reason_summary,
        concat(
            'release_events=',       CAST(quality_metrics['release_events'] AS STRING),
            ' anchors=',             CAST(quality_metrics['anchors'] AS STRING),
            ' prerelease=',          CAST(quality_metrics['prerelease_anchors'] AS STRING),
            ' baseline_zero=',       CAST(quality_metrics['baseline_zero_rows'] AS STRING),
            ' without_activity=',    CAST(quality_metrics['rows_without_activity'] AS STRING),
            ' manifest_hours=',      CAST(quality_metrics['manifest_hours'] AS STRING),
            ' complete_hours=',      CAST(quality_metrics['complete_hours'] AS STRING)
        ) AS metrics_summary
    FROM {TEST_AUDIT}
""")
audit_summary.createOrReplaceTempView("audit_summary")

## T3. Run on the empty tables

In [0]:
%run ../05_release_activity_shift $test_schema="test_gold_g5" $source_date="" $run_id="test-dry-run"

## T4. Checks: no input

The Silver tables have no release event. The run must end as `NO_INPUT`, publish nothing and still write its audit row. The run has created the Gold test table, without rows. The manifest is empty too, so it has 0 hours.

The first ten checks compare the table names the notebook used with the names of the test tables.

In every check table, `<=>` compares two values and also treats NULL and NULL as equal.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 the notebook read the test release table' AS check_name,
               '{RELEASE_TABLE}' AS actual, '{TEST_RELEASE}' AS expected
        UNION ALL
        SELECT '02 the notebook read the test push table', '{PUSH_TABLE}', '{TEST_PUSH}'
        UNION ALL
        SELECT '03 the notebook read the test pull request table', '{PULL_REQUEST_TABLE}', '{TEST_PULL_REQUEST}'
        UNION ALL
        SELECT '04 the notebook read the test issue table', '{ISSUE_TABLE}', '{TEST_ISSUE}'
        UNION ALL
        SELECT '05 the notebook read the test watch table', '{WATCH_TABLE}', '{TEST_WATCH}'
        UNION ALL
        SELECT '06 the notebook read the test fork table', '{FORK_TABLE}', '{TEST_FORK}'
        UNION ALL
        SELECT '07 the notebook read the test manifest', '{MANIFEST_TABLE}', '{TEST_MANIFEST}'
        UNION ALL
        SELECT '08 the notebook wrote the test Gold table', '{GOLD_TABLE}', '{TEST_GOLD}'
        UNION ALL
        SELECT '09 the notebook wrote the test staging table', '{STAGING_TABLE}', '{TEST_STAGING}'
        UNION ALL
        SELECT '10 the notebook wrote the test audit table', '{AUDIT_TABLE}', '{TEST_AUDIT}'
        UNION ALL
        SELECT '11 Gold rows', CAST(COUNT(*) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '12 audit rows', CAST(COUNT(*) AS STRING), '1' FROM {TEST_AUDIT}
        UNION ALL
        SELECT '13 audit status', MAX(status), 'NO_INPUT'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '14 audit pipeline_name', MAX(pipeline_name), 'gold_release_activity_shift'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '15 audit gold_table', MAX(gold_table), '{TEST_GOLD}'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '16 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '17 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '18 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '19 audit number of affected dates', CAST(MAX(size(affected_dates)) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '20 audit Silver rows read', MAX(source_summary), 'releases=0 push=0 pull_requests=0 issues=0 watches=0 forks=0'
        FROM audit_summary WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '21 audit reasons', MAX(reason_summary), 'not_published=0 draft=0 no_published_at=0 prerelease_unknown=0 no_release_id=0 repeated=0 not_covered=0'
        FROM audit_summary WHERE run_id = 'test-dry-run'
        UNION ALL
        SELECT '22 audit quality_metrics', MAX(metrics_summary), 'release_events=0 anchors=0 prerelease=0 baseline_zero=0 without_activity=0 manifest_hours=0 complete_hours=0'
        FROM audit_summary WHERE run_id = 'test-dry-run'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T4"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T4: {failed_count} check(s) failed. See the table above.")
print("T4: ALL CHECKS PASSED")

## T5. Load the hand-made rows

The rows are loaded in three cells:

| Cell | Rows | What it is |
|---|---|---|
| T5.1 | 168 | The manifest: every hour of the week. All complete except 5 February 20:00 |
| T5.2 | 29 | Release events. Event IDs ...01 to ...29 |
| T5.3 | 32 | Activity events of repositories 100, 200, 400, 410 and 999. Event IDs ...101 to ...132 |

Every event ID starts with `55000000` and exists only once in the eight tables together.

### T5.1 The manifest

`range(168)` gives the numbers 0 to 167. `timestampadd(HOUR, id, ...)` adds that many hours to 1 February 00:00, so the query gives the 168 hours of the week. Only the columns G5 reads are filled.

Then one hour is marked as not complete: 5 February 20:00. Rule 7 must leave out every release whose window touches it.

In [0]:
spark.sql(f"""
    INSERT INTO {TEST_MANIFEST} (source_hour_utc, source_date, is_complete)
    SELECT
        timestampadd(HOUR, id, TIMESTAMP'2024-02-01 00:00:00')                   AS source_hour_utc,
        CAST(timestampadd(HOUR, id, TIMESTAMP'2024-02-01 00:00:00') AS DATE)     AS source_date,
        true                                                                     AS is_complete
    FROM range(168)
""")

spark.sql(f"""
    UPDATE {TEST_MANIFEST}
    SET is_complete = false
    WHERE source_hour_utc = TIMESTAMP'2024-02-05 20:00:00'
""")

display(spark.sql(f"""
    SELECT
        COUNT(*)                        AS hours,
        count_if(is_complete = true)    AS complete_hours,
        MIN(source_hour_utc)            AS first_hour,
        MAX(source_hour_utc)            AS last_hour
    FROM {TEST_MANIFEST}
"""))

### T5.2 The release events

r is the published time. Most events come a few seconds after it, as in the real data.

**The rules 1 to 6.** Repositories 100 to 410.

| Event | Repository, release | What it is | Result |
|---|---|---|---|
| ...01 | 100, 1001 | r 4 Feb 12:00:00, tag `v1.0` | Anchor |
| ...02 | 100, 1002 | r 4 Feb 18:30:00, a prerelease, the event has the new name `org/alpha-renamed` | Anchor |
| ...03 | 200, 2001 | r 4 Feb 09:00:00, event at 09:00:02. It has the smaller event ID, but it is the later event | `REPEATED_RELEASE_EVENT` |
| ...04 | 200, 2001 | r 3 Feb 08:00:00, event at 08:00:05. The same rolling release, published a day earlier | Anchor |
| ...05 | 200, 2002 | r 4 Feb 10:00:00, tag `nightly-a` | Anchor: the same event time as ...06, and the smaller event ID |
| ...06 | 200, 2002 | The same event time, tag `nightly-b`, r 4 Feb 09:59:00. Its published time is earlier, but rule 6 orders by the event time | `REPEATED_RELEASE_EVENT` |
| ...07 | 300, 3001 | A draft, without a published time, at 09:00. It is earlier than ...08 | `DRAFT_OR_UNKNOWN` |
| ...08 | 300, 3001 | The published release, r 3 Feb 10:00:00. The draft does not take its place | Anchor |
| ...09 | 300, 3002 | The draft flag is unknown | `DRAFT_OR_UNKNOWN` |
| ...10 | 300, 3003 | Action `created`, a draft, no published time: it breaks rules 1, 2 and 3 | `NOT_PUBLISHED` |
| ...11 | 300, 3004 | The action is unknown | `NOT_PUBLISHED` |
| ...12 | 300, 3005 | Not a draft, but no published time | `MISSING_PUBLISHED_AT` |
| ...13 | 300, 3006 | A draft without a published time: rules 2 and 3 | `DRAFT_OR_UNKNOWN` |
| ...14 | 300, 3007 | The prerelease flag is unknown | `PRERELEASE_UNKNOWN` |
| ...15 | 300, none | No release ID | `MISSING_RELEASE_ID` |
| ...16 | 400, 4001 | r 4 Feb 06:00:00 | Anchor |
| ...17 | 410, 1001 | The release ID 1001 of repository 100, in another repository. r 3 Feb 15:00:30, not on the hour | Anchor |

**Rule 7.** Repository 500, without any activity. The manifest has the hours from 1 February 00:00 to 7 February 23:00, and 5 February 20:00 is not complete.

| Event | Release | r | Its window touches | Result |
|---|---|---|---|---|
| ...18 | 5001 | 2 Feb 00:00:00 | 1 Feb 00:00 to 2 Feb 23:00: 48 hours, the first hours of the week | Anchor |
| ...19 | 5002 | 1 Feb 23:59:59 | 31 Jan 23:00, which is not in the manifest | `WINDOW_NOT_COVERED` |
| ...20 | 5003 | 7 Feb 00:00:00 | 6 Feb 00:00 to 7 Feb 23:00: 48 hours, the last hours of the week | Anchor |
| ...21 | 5004 | 7 Feb 00:00:01 | 8 Feb 00:00, which is not in the manifest | `WINDOW_NOT_COVERED` |
| ...22 | 5005 | 4 Feb 20:00:00 | 3 Feb 20:00 to 5 Feb 19:00. The window ends just before the incomplete hour | Anchor |
| ...23 | 5006 | 4 Feb 20:00:01 | Up to 5 Feb 20:00, the incomplete hour | `WINDOW_NOT_COVERED` |
| ...24 | 5007 | 6 Feb 21:00:00, a prerelease | 5 Feb 21:00 to 7 Feb 20:00. The window starts just after the incomplete hour | Anchor |
| ...25 | 5008 | 6 Feb 20:59:59 | From 5 Feb 20:00, the incomplete hour | `WINDOW_NOT_COVERED` |
| ...26 | 5009 | 1 Feb 08:00:00 | Hours of 31 January | `WINDOW_NOT_COVERED` |
| ...27 | 5010 | 7 Feb 18:00:00 | Hours of 8 February | `WINDOW_NOT_COVERED` |
| ...28 | 5011 | 30 Jan 12:00:00. The **event** is on 2 Feb 10:00, inside the week | Hours of January. The window uses r, not the event time | `WINDOW_NOT_COVERED` |
| ...29 | 5012 | 5 Feb 12:00:00 | 5 Feb 20:00, the incomplete hour, in the after window | `WINDOW_NOT_COVERED` |

In all: 11 anchors. Left out: 2 `NOT_PUBLISHED`, 3 `DRAFT_OR_UNKNOWN`, 1 `MISSING_PUBLISHED_AT`, 1 `PRERELEASE_UNKNOWN`, 1 `MISSING_RELEASE_ID`, 2 `REPEATED_RELEASE_EVENT` and 8 `WINDOW_NOT_COVERED`. 11 + 18 = 29.

In [0]:
spark.sql("INSERT INTO " + TEST_RELEASE + " " + RELEASE_COLUMNS + """
VALUES
    -- event_id,     event_timestamp,                event_date,   repo, repo_name,           action,      release_id, tag,       published_at,                   draft, prerelease

    -- Repository 100: two releases
    ('55000000001', TIMESTAMP'2024-02-04 12:00:05', DATE'2024-02-04', 100, 'org/alpha',         'published', 1001, 'v1.0',      TIMESTAMP'2024-02-04 12:00:00', false, false),
    ('55000000002', TIMESTAMP'2024-02-04 18:30:05', DATE'2024-02-04', 100, 'org/alpha-renamed', 'published', 1002, 'v1.1-rc1',  TIMESTAMP'2024-02-04 18:30:00', false, true),

    -- Repository 200: two events of release 2001, two events of release 2002 at the same time
    ('55000000003', TIMESTAMP'2024-02-04 09:00:02', DATE'2024-02-04', 200, 'org/beta',          'published', 2001, 'nightly',   TIMESTAMP'2024-02-04 09:00:00', false, false),
    ('55000000004', TIMESTAMP'2024-02-03 08:00:05', DATE'2024-02-03', 200, 'org/beta',          'published', 2001, 'nightly',   TIMESTAMP'2024-02-03 08:00:00', false, false),
    ('55000000005', TIMESTAMP'2024-02-04 10:00:03', DATE'2024-02-04', 200, 'org/beta',          'published', 2002, 'nightly-a', TIMESTAMP'2024-02-04 10:00:00', false, false),
    ('55000000006', TIMESTAMP'2024-02-04 10:00:03', DATE'2024-02-04', 200, 'org/beta',          'published', 2002, 'nightly-b', TIMESTAMP'2024-02-04 09:59:00', false, false),

    -- Repository 300: the rules 1 to 5
    ('55000000007', TIMESTAMP'2024-02-03 09:00:00', DATE'2024-02-03', 300, 'org/gamma',         'published', 3001, 'v3.0',      NULL,                           true,  false),
    ('55000000008', TIMESTAMP'2024-02-03 10:00:05', DATE'2024-02-03', 300, 'org/gamma',         'published', 3001, 'v3.0',      TIMESTAMP'2024-02-03 10:00:00', false, false),
    ('55000000009', TIMESTAMP'2024-02-03 11:00:05', DATE'2024-02-03', 300, 'org/gamma',         'published', 3002, 'v3.1',      TIMESTAMP'2024-02-03 11:00:00', NULL,  false),
    ('55000000010', TIMESTAMP'2024-02-03 11:30:00', DATE'2024-02-03', 300, 'org/gamma',         'created',   3003, 'v3.2',      NULL,                           true,  false),
    ('55000000011', TIMESTAMP'2024-02-03 12:00:05', DATE'2024-02-03', 300, 'org/gamma',         NULL,        3004, 'v3.3',      TIMESTAMP'2024-02-03 12:00:00', false, false),
    ('55000000012', TIMESTAMP'2024-02-03 13:00:00', DATE'2024-02-03', 300, 'org/gamma',         'published', 3005, 'v3.4',      NULL,                           false, false),
    ('55000000013', TIMESTAMP'2024-02-03 13:30:00', DATE'2024-02-03', 300, 'org/gamma',         'published', 3006, 'v3.5',      NULL,                           true,  false),
    ('55000000014', TIMESTAMP'2024-02-03 14:00:05', DATE'2024-02-03', 300, 'org/gamma',         'published', 3007, 'v3.6',      TIMESTAMP'2024-02-03 14:00:00', false, NULL),
    ('55000000015', TIMESTAMP'2024-02-03 14:30:05', DATE'2024-02-03', 300, 'org/gamma',         'published', NULL, 'v3.7',      TIMESTAMP'2024-02-03 14:30:00', false, false),

    -- Repositories 400 and 410
    ('55000000016', TIMESTAMP'2024-02-04 06:00:05', DATE'2024-02-04', 400, 'org/delta',         'published', 4001, 'v4.0',      TIMESTAMP'2024-02-04 06:00:00', false, false),
    ('55000000017', TIMESTAMP'2024-02-03 15:00:35', DATE'2024-02-03', 410, 'org/epsilon',       'published', 1001, 'v1.0',      TIMESTAMP'2024-02-03 15:00:30', false, false),

    -- Repository 500: rule 7
    ('55000000018', TIMESTAMP'2024-02-02 00:00:05', DATE'2024-02-02', 500, 'org/zeta',          'published', 5001, 'r5001',     TIMESTAMP'2024-02-02 00:00:00', false, false),
    ('55000000019', TIMESTAMP'2024-02-02 00:00:04', DATE'2024-02-02', 500, 'org/zeta',          'published', 5002, 'r5002',     TIMESTAMP'2024-02-01 23:59:59', false, false),
    ('55000000020', TIMESTAMP'2024-02-07 00:00:05', DATE'2024-02-07', 500, 'org/zeta',          'published', 5003, 'r5003',     TIMESTAMP'2024-02-07 00:00:00', false, false),
    ('55000000021', TIMESTAMP'2024-02-07 00:00:06', DATE'2024-02-07', 500, 'org/zeta',          'published', 5004, 'r5004',     TIMESTAMP'2024-02-07 00:00:01', false, false),
    ('55000000022', TIMESTAMP'2024-02-04 20:00:05', DATE'2024-02-04', 500, 'org/zeta',          'published', 5005, 'r5005',     TIMESTAMP'2024-02-04 20:00:00', false, false),
    ('55000000023', TIMESTAMP'2024-02-04 20:00:06', DATE'2024-02-04', 500, 'org/zeta',          'published', 5006, 'r5006',     TIMESTAMP'2024-02-04 20:00:01', false, false),
    ('55000000024', TIMESTAMP'2024-02-06 21:00:05', DATE'2024-02-06', 500, 'org/zeta',          'published', 5007, 'r5007',     TIMESTAMP'2024-02-06 21:00:00', false, true),
    ('55000000025', TIMESTAMP'2024-02-06 21:00:04', DATE'2024-02-06', 500, 'org/zeta',          'published', 5008, 'r5008',     TIMESTAMP'2024-02-06 20:59:59', false, false),
    ('55000000026', TIMESTAMP'2024-02-01 08:00:05', DATE'2024-02-01', 500, 'org/zeta',          'published', 5009, 'r5009',     TIMESTAMP'2024-02-01 08:00:00', false, false),
    ('55000000027', TIMESTAMP'2024-02-07 18:00:05', DATE'2024-02-07', 500, 'org/zeta',          'published', 5010, 'r5010',     TIMESTAMP'2024-02-07 18:00:00', false, false),
    ('55000000028', TIMESTAMP'2024-02-02 10:00:00', DATE'2024-02-02', 500, 'org/zeta',          'published', 5011, 'r5011',     TIMESTAMP'2024-01-30 12:00:00', false, false),
    ('55000000029', TIMESTAMP'2024-02-05 12:00:05', DATE'2024-02-05', 500, 'org/zeta',          'published', 5012, 'r5012',     TIMESTAMP'2024-02-05 12:00:00', false, false)
""")

print("Release events loaded:", spark.table(TEST_RELEASE).count())

### T5.3 The activity events

**Repository 100 has two releases, so their windows overlap.** Release 1001: r = 4 Feb 12:00, its windows run from 3 Feb 12:00 to 5 Feb 12:00. Release 1002: r = 4 Feb 18:30, its windows run from 3 Feb 18:30 to 5 Feb 18:30.

| Event | Kind | Time | For release 1001 | For release 1002 |
|---|---|---|---|---|
| ...105 | Watch | 3 Feb 11:59:59 | Outside, 1 second before the start | Outside |
| ...101 | Push, 2 commits | 3 Feb 12:00:00 | Before, exactly at the start | Outside |
| ...112 | Fork | 3 Feb 18:29:59 | Before | Outside, 1 second before the start |
| ...111 | Pull request | 3 Feb 18:30:00 | Before | Before, exactly at the start |
| ...102 | Push, commit count unknown | 4 Feb 11:59:59 | Before, the last second | Before |
| ...103 | Pull request | 4 Feb 12:00:00 | After, exactly at r | Before |
| ...108 | Watch | 4 Feb 18:29:59 | After | Before, the last second |
| ...107 | Push, 3 commits | 4 Feb 18:30:00 | After | After, exactly at r |
| ...104 | Issue | 5 Feb 11:59:59 | After, the last second | After |
| ...106 | Fork | 5 Feb 12:00:00 | Outside, exactly at the end | After |
| ...114 | Issue | 5 Feb 15:00:00 | Outside | After |
| ...109 | Push, 0 commits | 5 Feb 18:29:59 | Outside | After, the last second |
| ...110 | Push, 5 commits | 5 Feb 18:30:00 | Outside | Outside, exactly at the end |
| ...113 | Push, 1 commit | No event time | Nowhere | Nowhere |
| ...115 | Issue comment | 4 Feb 13:00:00 | Not read | Not read |
| ...132 | Review | 4 Feb 14:00:00 | Not read | Not read |

**The other repositories.**

| Event | Repository | Kind | Time | Where it lands |
|---|---|---|---|---|
| ...116 | 999 | Push, 1 commit | 4 Feb 13:00:00 | Nowhere: repository 999 has no release |
| ...117 | 999 | Watch | 4 Feb 14:00:00 | Nowhere |
| ...118 | 200 | Watch | 3 Feb 07:00:00 | Release 2001 (r 3 Feb 08:00): before |
| ...119 | 200 | Push, 7 commits | 4 Feb 09:30:00 | Release 2001: outside, its windows end at 4 Feb 08:00. Release 2002 (r 4 Feb 10:00): before |
| ...125 | 400 | Issue | 3 Feb 05:59:59 | Release 4001 (r 4 Feb 06:00): outside, 1 second before the start |
| ...120 | 400 | Push, 1 commit | 4 Feb 06:00:00 | After, exactly at r |
| ...122 | 400 | Pull request | 4 Feb 07:00:00 | After |
| ...121 | 400 | Push, 4 commits | 4 Feb 20:00:00 | After |
| ...124 | 400 | Fork | 4 Feb 23:00:00 | After |
| ...123 | 400 | Watch | 5 Feb 05:59:59 | After, the last second |
| ...130 | 410 | Push, 1 commit | 2 Feb 15:00:29 | Release 1001 of repository 410 (r 3 Feb 15:00:30): outside, 1 second before the start |
| ...126 | 410 | Issue | 2 Feb 15:00:30 | Before, exactly at the start |
| ...127 | 410 | Push, 10 commits | 3 Feb 01:00:00 | Before |
| ...128 | 410 | Fork | 3 Feb 15:00:29 | Before, the last second |
| ...129 | 410 | Watch | 4 Feb 15:00:29 | After, the last second |
| ...131 | 410 | Pull request | 4 Feb 15:00:30 | Outside, exactly at the end |

In [0]:
spark.sql("INSERT INTO " + TEST_PUSH + " " + PUSH_COLUMNS + """
VALUES
    -- Repository 100
    ('55000000101', TIMESTAMP'2024-02-03 12:00:00', DATE'2024-02-03', 100, 'org/alpha',   2),
    ('55000000102', TIMESTAMP'2024-02-04 11:59:59', DATE'2024-02-04', 100, 'org/alpha',   NULL),
    ('55000000107', TIMESTAMP'2024-02-04 18:30:00', DATE'2024-02-04', 100, 'org/alpha',   3),
    ('55000000109', TIMESTAMP'2024-02-05 18:29:59', DATE'2024-02-05', 100, 'org/alpha',   0),
    ('55000000110', TIMESTAMP'2024-02-05 18:30:00', DATE'2024-02-05', 100, 'org/alpha',   5),
    ('55000000113', NULL,                           NULL,             100, 'org/alpha',   1),
    -- Repository 999, without a release
    ('55000000116', TIMESTAMP'2024-02-04 13:00:00', DATE'2024-02-04', 999, 'org/other',   1),
    -- Repository 200
    ('55000000119', TIMESTAMP'2024-02-04 09:30:00', DATE'2024-02-04', 200, 'org/beta',    7),
    -- Repository 400
    ('55000000120', TIMESTAMP'2024-02-04 06:00:00', DATE'2024-02-04', 400, 'org/delta',   1),
    ('55000000121', TIMESTAMP'2024-02-04 20:00:00', DATE'2024-02-04', 400, 'org/delta',   4),
    -- Repository 410
    ('55000000127', TIMESTAMP'2024-02-03 01:00:00', DATE'2024-02-03', 410, 'org/epsilon', 10),
    ('55000000130', TIMESTAMP'2024-02-02 15:00:29', DATE'2024-02-02', 410, 'org/epsilon', 1)
""")

spark.sql("INSERT INTO " + TEST_PULL_REQUEST + " " + ACTIVITY_COLUMNS + """
VALUES
    ('55000000103', TIMESTAMP'2024-02-04 12:00:00', DATE'2024-02-04', 100, 'org/alpha'),
    ('55000000111', TIMESTAMP'2024-02-03 18:30:00', DATE'2024-02-03', 100, 'org/alpha'),
    ('55000000122', TIMESTAMP'2024-02-04 07:00:00', DATE'2024-02-04', 400, 'org/delta'),
    ('55000000131', TIMESTAMP'2024-02-04 15:00:30', DATE'2024-02-04', 410, 'org/epsilon')
""")

spark.sql("INSERT INTO " + TEST_ISSUE + " " + ACTIVITY_COLUMNS + """
VALUES
    ('55000000104', TIMESTAMP'2024-02-05 11:59:59', DATE'2024-02-05', 100, 'org/alpha'),
    ('55000000114', TIMESTAMP'2024-02-05 15:00:00', DATE'2024-02-05', 100, 'org/alpha'),
    ('55000000125', TIMESTAMP'2024-02-03 05:59:59', DATE'2024-02-03', 400, 'org/delta'),
    ('55000000126', TIMESTAMP'2024-02-02 15:00:30', DATE'2024-02-02', 410, 'org/epsilon')
""")

spark.sql("INSERT INTO " + TEST_WATCH + " " + ACTIVITY_COLUMNS + """
VALUES
    ('55000000105', TIMESTAMP'2024-02-03 11:59:59', DATE'2024-02-03', 100, 'org/alpha'),
    ('55000000108', TIMESTAMP'2024-02-04 18:29:59', DATE'2024-02-04', 100, 'org/alpha'),
    ('55000000117', TIMESTAMP'2024-02-04 14:00:00', DATE'2024-02-04', 999, 'org/other'),
    ('55000000118', TIMESTAMP'2024-02-03 07:00:00', DATE'2024-02-03', 200, 'org/beta'),
    ('55000000123', TIMESTAMP'2024-02-05 05:59:59', DATE'2024-02-05', 400, 'org/delta'),
    ('55000000129', TIMESTAMP'2024-02-04 15:00:29', DATE'2024-02-04', 410, 'org/epsilon')
""")

spark.sql("INSERT INTO " + TEST_FORK + " " + ACTIVITY_COLUMNS + """
VALUES
    ('55000000106', TIMESTAMP'2024-02-05 12:00:00', DATE'2024-02-05', 100, 'org/alpha'),
    ('55000000112', TIMESTAMP'2024-02-03 18:29:59', DATE'2024-02-03', 100, 'org/alpha'),
    ('55000000124', TIMESTAMP'2024-02-04 23:00:00', DATE'2024-02-04', 400, 'org/delta'),
    ('55000000128', TIMESTAMP'2024-02-03 15:00:29', DATE'2024-02-03', 410, 'org/epsilon')
""")

# Not G5 sources: these rows must not be counted.
spark.sql("INSERT INTO " + TEST_ISSUE_COMMENT + " " + ACTIVITY_COLUMNS + """
VALUES
    ('55000000115', TIMESTAMP'2024-02-04 13:00:00', DATE'2024-02-04', 100, 'org/alpha')
""")

spark.sql("INSERT INTO " + TEST_REVIEW + " " + ACTIVITY_COLUMNS + """
VALUES
    ('55000000132', TIMESTAMP'2024-02-04 14:00:00', DATE'2024-02-04', 100, 'org/alpha')
""")

print("Rows in the Silver test tables:")
for table_name in SILVER_TABLE_NAMES:
    print("  ", table_name, ":", spark.table(TEST_SILVER + "." + table_name).count())

### A three-line summary of every Gold row

The checks below compare whole Gold rows. This view writes a row into three readable lines, for example:

`event=55000000001 r=2024-02-04 12:00:00 tag=v1.0 pre=false name=org/alpha`

`before=4 after=4 change=0 pct=0.0 zero=false`

`push=2/1 pr=1/1 issue=0/1 watch=0/1 fork=1/0 eligible=1/1 commits=2/3`

- The first line is the anchor: its event, r, the tag, the prerelease flag and the name.
- The second line is the totals: before, after, the change, the percent change and the baseline flag.
- The third line is the five kinds and the push commits, each as before/after. `push=2/1` means 2 pushes before and 1 push after.

A NULL value is written as `NULL`. The view is not a copy. It always reads the Gold table as it is at that moment.

In [0]:
gold_summary = spark.sql(f"""
    SELECT
        repo_id,
        release_anchor_key,
        concat(
            'event=',     anchor_event_id,
            ' r=',        CAST(release_published_at AS STRING),
            ' tag=',      COALESCE(release_tag, 'NULL'),
            ' pre=',      CAST(is_prerelease AS STRING),
            ' name=',     COALESCE(repo_name, 'NULL')
        ) AS anchor_summary,
        concat(
            'before=',    CAST(activity_events_before AS STRING),
            ' after=',    CAST(activity_events_after AS STRING),
            ' change=',   CAST(activity_events_change AS STRING),
            ' pct=',      COALESCE(CAST(activity_events_pct_change AS STRING), 'NULL'),
            ' zero=',     CAST(baseline_zero_flag AS STRING)
        ) AS count_summary,
        concat(
            'push=',      CAST(push_events_before AS STRING),  '/', CAST(push_events_after AS STRING),
            ' pr=',       CAST(pr_events_before AS STRING),    '/', CAST(pr_events_after AS STRING),
            ' issue=',    CAST(issue_events_before AS STRING), '/', CAST(issue_events_after AS STRING),
            ' watch=',    CAST(watch_events_before AS STRING), '/', CAST(watch_events_after AS STRING),
            ' fork=',     CAST(fork_events_before AS STRING),  '/', CAST(fork_events_after AS STRING),
            ' eligible=', CAST(push_commit_eligible_events_before AS STRING), '/', CAST(push_commit_eligible_events_after AS STRING),
            ' commits=',  CAST(known_push_commits_before AS STRING), '/', CAST(known_push_commits_after AS STRING)
        ) AS kind_summary
    FROM {TEST_GOLD}
""")
gold_summary.createOrReplaceTempView("gold_summary")

## T6. Run 1: all dates

In [0]:
%run ../05_release_activity_shift $test_schema="test_gold_g5" $source_date="" $run_id="test-run1"

## T7. Checks of run 1: schema and exact values

Worked out by hand, from the tables in T5.

**The run.** 29 release events, 11 anchors, so 11 rows. The activity tables have 12 pushes, 4 pull requests, 4 issues, 6 watches and 4 forks. The push ...113 has no event time and is counted as left out. The release events are on all seven days, so the audit row lists all seven days.

**Repository 100.**

- Release 1001. Before: the pushes ...101 and ...102, the pull request ...111 and the fork ...112, so 4. After: the pull request ...103, the issue ...104, the push ...107 and the watch ...108, so 4. Change 0, percent change 100 × 0 ÷ 4 = 0.0.
- Release 1001, the push commits. Before, only ...101 has a known count: 1 eligible push, 2 commits. After: ...107, 1 eligible push, 3 commits.
- Release 1002. Before: ...102, ...103, ...108 and ...111, so 4. After: the pushes ...107 and ...109, the issues ...104 and ...114, and the fork ...106, so 5. Change 1, percent change 100 × 1 ÷ 4 = 25.0.
- Release 1002, the push commits. Before, the only push ...102 has no known count: 0 eligible pushes and 0 commits, not NULL. After: ...107 with 3 and ...109 with 0, so 2 eligible pushes and 3 commits. A push with 0 commits is a known count.
- Six events lie in the windows of both releases. They are counted for each release.
- The issue comment ...115 and the review ...132 lie in the windows, but they are not counted.
- The label of release 1002 is `org/alpha-renamed`, the name on its own event.

**The other rows.**

| Row | Anchor event | Before | After | Change | Percent change | Baseline flag |
|---|---|---|---|---|---|---|
| 200, 2001 | ...04, r 3 Feb 08:00 | 1: the watch ...118 | 0 | −1 | −100.0 | false |
| 200, 2002 | ...05, tag `nightly-a` | 1: the push ...119, 7 commits | 0 | −1 | −100.0 | false |
| 300, 3001 | ...08 | 0 | 0 | 0 | NULL | true |
| 400, 4001 | ...16 | 0 | 5: 2 pushes (1 + 4 commits), a pull request, a watch and a fork | 5 | NULL | true |
| 410, 1001 | ...17, r 3 Feb 15:00:30 | 3: the issue ...126, the push ...127 (10 commits), the fork ...128 | 1: the watch ...129 | −2 | 100 × −2 ÷ 3 = −66.67 | false |
| 500, four rows | ...18, ...20, ...22, ...24 | 0 | 0 | 0 | NULL | true |

- Release 2001: if the event ...03 were the anchor (r 4 Feb 09:00), the watch ...118 would be outside and the push ...119 would be after. The row would be 0 before and 1 after. So the check proves that the earliest event is used.
- Release 4001 has events after but none before: the percent change is NULL, and the baseline flag is true.
- Release 1001 of repository 410 has the release ID of repository 100. The two rows do not share their activity.
- Repository 410: 49 hours are needed, because r is not on the hour. The window runs from 2 Feb 15:00:30 to 4 Feb 15:00:30.

**The totals.** 2 prerelease rows (1002 and 5007). 6 rows with 0 before: 3001, 4001 and the four of repository 500. 5 of them also have 0 after: all except 4001. The manifest has 168 hours, 167 of them complete.

In [0]:
actual_schema = spark.table(TEST_GOLD).schema.simpleString()
staging_schema = spark.table(TEST_STAGING).schema.simpleString()

expected_schema = (
    "struct<repo_id:bigint,release_anchor_key:string,repo_name:string,"
    "release_id:bigint,release_tag:string,release_published_at:timestamp,"
    "is_prerelease:boolean,anchor_event_id:string,"
    "window_start_at:timestamp,window_end_at:timestamp,"
    "activity_events_before:bigint,activity_events_after:bigint,activity_events_change:bigint,"
    "activity_events_pct_change:double,baseline_zero_flag:boolean,"
    "push_events_before:bigint,push_events_after:bigint,"
    "pr_events_before:bigint,pr_events_after:bigint,"
    "issue_events_before:bigint,issue_events_after:bigint,"
    "watch_events_before:bigint,watch_events_after:bigint,"
    "fork_events_before:bigint,fork_events_after:bigint,"
    "push_commit_eligible_events_before:bigint,push_commit_eligible_events_after:bigint,"
    "known_push_commits_before:bigint,known_push_commits_after:bigint,"
    "_gold_run_id:string,_gold_updated_at:timestamp>"
)

print("Gold    :", actual_schema)
print("Staging :", staging_schema)
print("Expected:", expected_schema)

# The three texts are long. The check table below gets a short text for each comparison.
if actual_schema == expected_schema:
    gold_schema_result = "as in the contract"
else:
    gold_schema_result = "DIFFERENT, see the lines above"

if staging_schema == expected_schema:
    staging_schema_result = "as in the contract"
else:
    staging_schema_result = "DIFFERENT, see the lines above"

Check 04 lists the keys of all rows in one text. `collect_list` puts the keys of all rows into one list, `sort_array` sorts it, and `concat_ws` writes it as one text with a space between the keys. Each key is written as repository:release.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 columns and types of the Gold table' AS check_name,
               '{gold_schema_result}' AS actual, 'as in the contract' AS expected
        UNION ALL
        SELECT '02 columns and types of the staging table', '{staging_schema_result}', 'as in the contract'
        UNION ALL
        SELECT '03 Gold rows', CAST(COUNT(*) AS STRING), '11' FROM {TEST_GOLD}
        UNION ALL
        SELECT '04 the keys of all rows',
               concat_ws(' ', sort_array(collect_list(concat(CAST(repo_id AS STRING), ':', release_anchor_key)))),
               '100:1001 100:1002 200:2001 200:2002 300:3001 400:4001 410:1001 500:5001 500:5003 500:5005 500:5007'
        FROM {TEST_GOLD}
        UNION ALL
        SELECT '05 rows written by this run', CAST(count_if(_gold_run_id = 'test-run1' AND _gold_updated_at IS NOT NULL) AS STRING), '11' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 rows whose key is not the release_id as text', CAST(count_if(release_anchor_key <> CAST(release_id AS STRING)) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '07 rows whose window is not r - 24 hours to r + 24 hours',
               CAST(count_if(window_start_at <> release_published_at - INTERVAL 24 HOURS
                          OR window_end_at <> release_published_at + INTERVAL 24 HOURS) AS STRING), '0'
        FROM {TEST_GOLD}
        UNION ALL
        SELECT '08 rows with is_prerelease NULL', CAST(count_if(is_prerelease IS NULL) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '09 repository 100, release 1001: anchor', MAX(anchor_summary), 'event=55000000001 r=2024-02-04 12:00:00 tag=v1.0 pre=false name=org/alpha'
        FROM gold_summary WHERE repo_id = 100 AND release_anchor_key = '1001'
        UNION ALL
        SELECT '10 repository 100, release 1001: totals', MAX(count_summary), 'before=4 after=4 change=0 pct=0.0 zero=false'
        FROM gold_summary WHERE repo_id = 100 AND release_anchor_key = '1001'
        UNION ALL
        SELECT '11 repository 100, release 1001: kinds', MAX(kind_summary), 'push=2/1 pr=1/1 issue=0/1 watch=0/1 fork=1/0 eligible=1/1 commits=2/3'
        FROM gold_summary WHERE repo_id = 100 AND release_anchor_key = '1001'
        UNION ALL
        SELECT '12 repository 100, release 1002: anchor', MAX(anchor_summary), 'event=55000000002 r=2024-02-04 18:30:00 tag=v1.1-rc1 pre=true name=org/alpha-renamed'
        FROM gold_summary WHERE repo_id = 100 AND release_anchor_key = '1002'
        UNION ALL
        SELECT '13 repository 100, release 1002: totals', MAX(count_summary), 'before=4 after=5 change=1 pct=25.0 zero=false'
        FROM gold_summary WHERE repo_id = 100 AND release_anchor_key = '1002'
        UNION ALL
        SELECT '14 repository 100, release 1002: kinds', MAX(kind_summary), 'push=1/2 pr=2/0 issue=0/2 watch=1/0 fork=0/1 eligible=0/2 commits=0/3'
        FROM gold_summary WHERE repo_id = 100 AND release_anchor_key = '1002'
        UNION ALL
        SELECT '15 repository 200, release 2001: anchor is the earliest event', MAX(anchor_summary), 'event=55000000004 r=2024-02-03 08:00:00 tag=nightly pre=false name=org/beta'
        FROM gold_summary WHERE repo_id = 200 AND release_anchor_key = '2001'
        UNION ALL
        SELECT '16 repository 200, release 2001: totals', MAX(count_summary), 'before=1 after=0 change=-1 pct=-100.0 zero=false'
        FROM gold_summary WHERE repo_id = 200 AND release_anchor_key = '2001'
        UNION ALL
        SELECT '17 repository 200, release 2001: kinds', MAX(kind_summary), 'push=0/0 pr=0/0 issue=0/0 watch=1/0 fork=0/0 eligible=0/0 commits=0/0'
        FROM gold_summary WHERE repo_id = 200 AND release_anchor_key = '2001'
        UNION ALL
        SELECT '18 repository 200, release 2002: anchor is the smaller event ID', MAX(anchor_summary), 'event=55000000005 r=2024-02-04 10:00:00 tag=nightly-a pre=false name=org/beta'
        FROM gold_summary WHERE repo_id = 200 AND release_anchor_key = '2002'
        UNION ALL
        SELECT '19 repository 200, release 2002: totals', MAX(count_summary), 'before=1 after=0 change=-1 pct=-100.0 zero=false'
        FROM gold_summary WHERE repo_id = 200 AND release_anchor_key = '2002'
        UNION ALL
        SELECT '20 repository 200, release 2002: kinds', MAX(kind_summary), 'push=1/0 pr=0/0 issue=0/0 watch=0/0 fork=0/0 eligible=1/0 commits=7/0'
        FROM gold_summary WHERE repo_id = 200 AND release_anchor_key = '2002'
        UNION ALL
        SELECT '21 repository 300, release 3001: anchor is not the draft', MAX(anchor_summary), 'event=55000000008 r=2024-02-03 10:00:00 tag=v3.0 pre=false name=org/gamma'
        FROM gold_summary WHERE repo_id = 300 AND release_anchor_key = '3001'
        UNION ALL
        SELECT '22 repository 300, release 3001: totals (no activity)', MAX(count_summary), 'before=0 after=0 change=0 pct=NULL zero=true'
        FROM gold_summary WHERE repo_id = 300 AND release_anchor_key = '3001'
        UNION ALL
        SELECT '23 repository 400, release 4001: totals (0 before)', MAX(count_summary), 'before=0 after=5 change=5 pct=NULL zero=true'
        FROM gold_summary WHERE repo_id = 400 AND release_anchor_key = '4001'
        UNION ALL
        SELECT '24 repository 400, release 4001: kinds', MAX(kind_summary), 'push=0/2 pr=0/1 issue=0/0 watch=0/1 fork=0/1 eligible=0/2 commits=0/5'
        FROM gold_summary WHERE repo_id = 400 AND release_anchor_key = '4001'
        UNION ALL
        SELECT '25 repository 410, release 1001: anchor', MAX(anchor_summary), 'event=55000000017 r=2024-02-03 15:00:30 tag=v1.0 pre=false name=org/epsilon'
        FROM gold_summary WHERE repo_id = 410 AND release_anchor_key = '1001'
        UNION ALL
        SELECT '26 repository 410, release 1001: totals', MAX(count_summary), 'before=3 after=1 change=-2 pct=-66.67 zero=false'
        FROM gold_summary WHERE repo_id = 410 AND release_anchor_key = '1001'
        UNION ALL
        SELECT '27 repository 410, release 1001: kinds', MAX(kind_summary), 'push=1/0 pr=0/0 issue=1/0 watch=0/1 fork=1/0 eligible=1/0 commits=10/0'
        FROM gold_summary WHERE repo_id = 410 AND release_anchor_key = '1001'
        UNION ALL
        SELECT '28 repository 410, release 1001: window',
               MAX(concat(CAST(window_start_at AS STRING), ' to ', CAST(window_end_at AS STRING))),
               '2024-02-02 15:00:30 to 2024-02-04 15:00:30'
        FROM {TEST_GOLD} WHERE repo_id = 410 AND release_anchor_key = '1001'
        UNION ALL
        SELECT '29 repository 500: rows without any activity',
               CAST(count_if(activity_events_before = 0 AND activity_events_after = 0 AND activity_events_pct_change IS NULL AND baseline_zero_flag = true) AS STRING), '4'
        FROM {TEST_GOLD} WHERE repo_id = 500
        UNION ALL
        SELECT '30 repository 500, release 5007: anchor', MAX(anchor_summary), 'event=55000000024 r=2024-02-06 21:00:00 tag=r5007 pre=true name=org/zeta'
        FROM gold_summary WHERE repo_id = 500 AND release_anchor_key = '5007'
        UNION ALL
        SELECT '31 repository 500, release 5001: window, the first hours of the week',
               MAX(concat(CAST(window_start_at AS STRING), ' to ', CAST(window_end_at AS STRING))),
               '2024-02-01 00:00:00 to 2024-02-03 00:00:00'
        FROM {TEST_GOLD} WHERE repo_id = 500 AND release_anchor_key = '5001'
        UNION ALL
        SELECT '32 repository 500, release 5003: window, the last hours of the week',
               MAX(concat(CAST(window_start_at AS STRING), ' to ', CAST(window_end_at AS STRING))),
               '2024-02-06 00:00:00 to 2024-02-08 00:00:00'
        FROM {TEST_GOLD} WHERE repo_id = 500 AND release_anchor_key = '5003'
        UNION ALL
        SELECT '33 audit status', MAX(status), 'SUCCEEDED'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '34 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '35 audit source_date', CAST(MAX(source_date) AS STRING), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '36 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02, 2024-02-03, 2024-02-04, 2024-02-05, 2024-02-06, 2024-02-07]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '37 audit Silver rows read', MAX(source_summary), 'releases=29 push=12 pull_requests=4 issues=4 watches=6 forks=4'
        FROM audit_summary WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '38 audit reasons', MAX(reason_summary), 'not_published=2 draft=3 no_published_at=1 prerelease_unknown=1 no_release_id=1 repeated=2 not_covered=8'
        FROM audit_summary WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '39 audit quality_metrics', MAX(metrics_summary), 'release_events=29 anchors=11 prerelease=2 baseline_zero=6 without_activity=5 manifest_hours=168 complete_hours=167'
        FROM audit_summary WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '40 audit rows without an event time', CAST(MAX(rows_excluded['missing_event_timestamp']) AS STRING), '1'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '41 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '11'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '42 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '43 audit failed_checks', MAX(failed_checks), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '44 audit error_message', MAX(error_message), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '45 audit has a table version', CAST(MAX(gold_table_version) IS NOT NULL AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '46 audit pipeline_name', MAX(pipeline_name), 'gold_release_activity_shift'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '47 audit gold_table', MAX(gold_table), '{TEST_GOLD}'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
        UNION ALL
        SELECT '48 audit run times are in order', CAST(MAX(run_finished_at >= run_started_at) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run1'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T7"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T7: {failed_count} check(s) failed. See the table above.")
print("T7: ALL CHECKS PASSED")

## T8. Run 2: the same run again, now with a source date

First the Gold rows of run 1 are saved as a copy, so that run 2 can be compared with it. The copy holds the key and the three summary lines of every row.

Run 2 gets `source_date = 2024-02-03`. G5 is rebuilt completely on every run, so the date changes nothing in the result. It is only written into the audit row.

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {TEST_COPY} AS
    SELECT repo_id, release_anchor_key, anchor_summary, count_summary, kind_summary
    FROM gold_summary
""")

print("Copy of the Gold rows after run 1:", TEST_COPY)

In [0]:
%run ../05_release_activity_shift $test_schema="test_gold_g5" $source_date="2024-02-03" $run_id="test-run2"

## T9. Checks of run 2: nothing changed

`EXCEPT` returns the rows of the first query that are not in the second one. Both directions must return no rows. The two technical columns are not part of the summary, because they are new in every run.

The scope is still `ALL_DATES`, the source date is in the audit row, and every row now carries the ID of run 2, also the rows of days other than 3 February.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '03 audit source_date', CAST(MAX(source_date) AS STRING), '2024-02-03'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '04 Gold rows', CAST(COUNT(*) AS STRING), '11' FROM {TEST_GOLD}
        UNION ALL
        SELECT '05 rows of run 2 that are not in run 1', CAST(COUNT(*) AS STRING), '0'
        FROM (
            SELECT repo_id, release_anchor_key, anchor_summary, count_summary, kind_summary FROM gold_summary
            EXCEPT
            SELECT repo_id, release_anchor_key, anchor_summary, count_summary, kind_summary FROM {TEST_COPY}
        )
        UNION ALL
        SELECT '06 rows of run 1 that are not in run 2', CAST(COUNT(*) AS STRING), '0'
        FROM (
            SELECT repo_id, release_anchor_key, anchor_summary, count_summary, kind_summary FROM {TEST_COPY}
            EXCEPT
            SELECT repo_id, release_anchor_key, anchor_summary, count_summary, kind_summary FROM gold_summary
        )
        UNION ALL
        SELECT '07 rows written by run 2', CAST(count_if(_gold_run_id = 'test-run2') AS STRING), '11' FROM {TEST_GOLD}
        UNION ALL
        SELECT '08 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '11'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '09 audit affected_dates', MAX(CAST(affected_dates AS STRING)), '[2024-02-01, 2024-02-02, 2024-02-03, 2024-02-04, 2024-02-05, 2024-02-06, 2024-02-07]'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '10 audit Silver rows read', MAX(source_summary), 'releases=29 push=12 pull_requests=4 issues=4 watches=6 forks=4'
        FROM audit_summary WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '11 audit reasons', MAX(reason_summary), 'not_published=2 draft=3 no_published_at=1 prerelease_unknown=1 no_release_id=1 repeated=2 not_covered=8'
        FROM audit_summary WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '12 audit quality_metrics', MAX(metrics_summary), 'release_events=29 anchors=11 prerelease=2 baseline_zero=6 without_activity=5 manifest_hours=168 complete_hours=167'
        FROM audit_summary WHERE run_id = 'test-run2'
        UNION ALL
        SELECT '13 different runs in the audit table so far', CAST(COUNT(DISTINCT run_id) AS STRING), '3' FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T9"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T9: {failed_count} check(s) failed. See the table above.")
print("T9: ALL CHECKS PASSED")

## T10. Run 3: late events

Four new Silver rows arrive:

| Event | Table | Repository | Time | What it tests |
|---|---|---|---|---|
| ...133 | Fork | 100 | 5 Feb 00:00:00 | A late event inside the after windows of **both** releases of repository 100. It must change two rows |
| ...30 | Release | 410 | r 6 Feb 22:00:00 | A new release, release 6001. Its windows cover 5 Feb 22:00 to 7 Feb 22:00, just after the incomplete hour. It adds a row |
| ...134 | Push, 2 commits | 410 | 7 Feb 10:00:00 | Activity after the new release |
| ...31 | Release | 400 | r 4 Feb 05:00:00, event at 05:00:05 | A late event of release 4001 that is **earlier** than the event ...16 (06:00:05). It becomes the anchor, and ...16 becomes a repeated event |

The run gets `source_date = 2024-02-06`. It still rebuilds the whole table.

In [0]:
spark.sql("INSERT INTO " + TEST_FORK + " " + ACTIVITY_COLUMNS + """
VALUES
    ('55000000133', TIMESTAMP'2024-02-05 00:00:00', DATE'2024-02-05', 100, 'org/alpha')
""")

spark.sql("INSERT INTO " + TEST_PUSH + " " + PUSH_COLUMNS + """
VALUES
    ('55000000134', TIMESTAMP'2024-02-07 10:00:00', DATE'2024-02-07', 410, 'org/epsilon', 2)
""")

spark.sql("INSERT INTO " + TEST_RELEASE + " " + RELEASE_COLUMNS + """
VALUES
    ('55000000030', TIMESTAMP'2024-02-06 22:00:05', DATE'2024-02-06', 410, 'org/epsilon',       'published', 6001, 'v6.0',      TIMESTAMP'2024-02-06 22:00:00', false, false),
    ('55000000031', TIMESTAMP'2024-02-04 05:00:05', DATE'2024-02-04', 400, 'org/delta',         'published', 4001, 'v4.0',      TIMESTAMP'2024-02-04 05:00:00', false, false)
""")

print("Rows in the Silver test tables:")
for table_name in SILVER_TABLE_NAMES:
    print("  ", table_name, ":", spark.table(TEST_SILVER + "." + table_name).count())

In [0]:
%run ../05_release_activity_shift $test_schema="test_gold_g5" $source_date="2024-02-06" $run_id="test-run3"

## T11. Checks of run 3

Worked out by hand.

- **Repository 100.** The fork ...133 is after both releases. Release 1001: 4 before, 5 after, change 1, percent change 100 × 1 ÷ 4 = 25.0. Release 1002: 4 before, 6 after, change 2, percent change 50.0.
- **Repository 400.** The anchor of release 4001 is now the event ...31, with r = 4 Feb 05:00. Its windows run from 3 Feb 05:00 to 5 Feb 05:00. The issue ...125 (3 Feb 05:59:59) is now **before**, and the watch ...123 (5 Feb 05:59:59) is now **outside**. Before: 1. After: the two pushes, the pull request and the fork, 4. Change 3, percent change 100 × 3 ÷ 1 = 300.0. A percent change can be above 100.
- **Repository 410.** The new release 6001: 0 before, 1 after (the push ...134, 2 commits). The release 1001 of repository 410 is not changed: the push ...134 is outside its windows.
- 12 rows. Compared with run 1, three rows have new values (1001, 1002 and 4001) and one row is new (6001). The other 8 rows are the same.
- 31 release events, 12 anchors, 3 repeated events (...03, ...06 and ...16).
- 6 rows with 0 before: 3001, 6001 and the four of repository 500. 5 of them have no activity at all.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '03 audit source_date', CAST(MAX(source_date) AS STRING), '2024-02-06'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '04 Gold rows', CAST(COUNT(*) AS STRING), '12' FROM {TEST_GOLD}
        UNION ALL
        SELECT '05 rows written by run 3', CAST(count_if(_gold_run_id = 'test-run3') AS STRING), '12' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 rows of run 3 that are not in run 1 (3 changed, 1 new)', CAST(COUNT(*) AS STRING), '4'
        FROM (
            SELECT repo_id, release_anchor_key, anchor_summary, count_summary, kind_summary FROM gold_summary
            EXCEPT
            SELECT repo_id, release_anchor_key, anchor_summary, count_summary, kind_summary FROM {TEST_COPY}
        )
        UNION ALL
        SELECT '07 rows of run 1 that are not in run 3 (3 changed)', CAST(COUNT(*) AS STRING), '3'
        FROM (
            SELECT repo_id, release_anchor_key, anchor_summary, count_summary, kind_summary FROM {TEST_COPY}
            EXCEPT
            SELECT repo_id, release_anchor_key, anchor_summary, count_summary, kind_summary FROM gold_summary
        )
        UNION ALL
        SELECT '08 repository 100, release 1001: totals', MAX(count_summary), 'before=4 after=5 change=1 pct=25.0 zero=false'
        FROM gold_summary WHERE repo_id = 100 AND release_anchor_key = '1001'
        UNION ALL
        SELECT '09 repository 100, release 1001: kinds', MAX(kind_summary), 'push=2/1 pr=1/1 issue=0/1 watch=0/1 fork=1/1 eligible=1/1 commits=2/3'
        FROM gold_summary WHERE repo_id = 100 AND release_anchor_key = '1001'
        UNION ALL
        SELECT '10 repository 100, release 1002: totals', MAX(count_summary), 'before=4 after=6 change=2 pct=50.0 zero=false'
        FROM gold_summary WHERE repo_id = 100 AND release_anchor_key = '1002'
        UNION ALL
        SELECT '11 repository 100, release 1002: kinds', MAX(kind_summary), 'push=1/2 pr=2/0 issue=0/2 watch=1/0 fork=0/2 eligible=0/2 commits=0/3'
        FROM gold_summary WHERE repo_id = 100 AND release_anchor_key = '1002'
        UNION ALL
        SELECT '12 repository 400, release 4001: the earlier event is the anchor now', MAX(anchor_summary), 'event=55000000031 r=2024-02-04 05:00:00 tag=v4.0 pre=false name=org/delta'
        FROM gold_summary WHERE repo_id = 400 AND release_anchor_key = '4001'
        UNION ALL
        SELECT '13 repository 400, release 4001: totals', MAX(count_summary), 'before=1 after=4 change=3 pct=300.0 zero=false'
        FROM gold_summary WHERE repo_id = 400 AND release_anchor_key = '4001'
        UNION ALL
        SELECT '14 repository 400, release 4001: kinds', MAX(kind_summary), 'push=0/2 pr=0/1 issue=1/0 watch=0/0 fork=0/1 eligible=0/2 commits=0/5'
        FROM gold_summary WHERE repo_id = 400 AND release_anchor_key = '4001'
        UNION ALL
        SELECT '15 repository 410, release 6001: anchor', MAX(anchor_summary), 'event=55000000030 r=2024-02-06 22:00:00 tag=v6.0 pre=false name=org/epsilon'
        FROM gold_summary WHERE repo_id = 410 AND release_anchor_key = '6001'
        UNION ALL
        SELECT '16 repository 410, release 6001: totals', MAX(count_summary), 'before=0 after=1 change=1 pct=NULL zero=true'
        FROM gold_summary WHERE repo_id = 410 AND release_anchor_key = '6001'
        UNION ALL
        SELECT '17 repository 410, release 6001: kinds', MAX(kind_summary), 'push=0/1 pr=0/0 issue=0/0 watch=0/0 fork=0/0 eligible=0/1 commits=0/2'
        FROM gold_summary WHERE repo_id = 410 AND release_anchor_key = '6001'
        UNION ALL
        SELECT '18 repository 410, release 1001: totals (unchanged)', MAX(count_summary), 'before=3 after=1 change=-2 pct=-66.67 zero=false'
        FROM gold_summary WHERE repo_id = 410 AND release_anchor_key = '1001'
        UNION ALL
        SELECT '19 audit Silver rows read', MAX(source_summary), 'releases=31 push=13 pull_requests=4 issues=4 watches=6 forks=5'
        FROM audit_summary WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '20 audit reasons', MAX(reason_summary), 'not_published=2 draft=3 no_published_at=1 prerelease_unknown=1 no_release_id=1 repeated=3 not_covered=8'
        FROM audit_summary WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '21 audit quality_metrics', MAX(metrics_summary), 'release_events=31 anchors=12 prerelease=2 baseline_zero=6 without_activity=5 manifest_hours=168 complete_hours=167'
        FROM audit_summary WHERE run_id = 'test-run3'
        UNION ALL
        SELECT '22 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '12'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run3'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T11"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T11: {failed_count} check(s) failed. See the table above.")
print("T11: ALL CHECKS PASSED")

## T12. Run 4: removed events

Three Silver rows are removed:

| Event | Table | What it tests |
|---|---|---|
| ...119 | Push | The only activity of release 2002. Its row goes down to 0 before and 0 after |
| ...04 | Release | The anchor of release 2001. The other event of that release, ...03, becomes the anchor |
| ...20 | Release | The only event of release 5003. Its row disappears |

The run is for all dates.

In [0]:
spark.sql(f"""
    DELETE FROM {TEST_PUSH}
    WHERE event_id = '55000000119'
""")

spark.sql(f"""
    DELETE FROM {TEST_RELEASE}
    WHERE event_id IN ('55000000004', '55000000020')
""")

print("Rows in the Silver test tables:")
for table_name in SILVER_TABLE_NAMES:
    print("  ", table_name, ":", spark.table(TEST_SILVER + "." + table_name).count())

In [0]:
%run ../05_release_activity_shift $test_schema="test_gold_g5" $source_date="" $run_id="test-run4"

## T13. Checks of run 4

Worked out by hand.

- Release 2001 has the anchor ...03 now, with r = 4 Feb 09:00. Its windows run from 3 Feb 09:00 to 5 Feb 09:00. The watch ...118 (3 Feb 07:00) is outside, and the push ...119 is gone. So 0 before and 0 after.
- Release 2002 has lost its only push: 0 before and 0 after.
- Release 5003 has no event any more, so it has no row.
- 11 rows. 29 release events, 11 anchors, 2 repeated events (...06 and ...16).
- 7 rows with 0 before: 2001, 2002, 3001, 6001 and three of repository 500. 6 of them have no activity at all.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '02 Gold rows', CAST(COUNT(*) AS STRING), '11' FROM {TEST_GOLD}
        UNION ALL
        SELECT '03 the keys of all rows',
               concat_ws(' ', sort_array(collect_list(concat(CAST(repo_id AS STRING), ':', release_anchor_key)))),
               '100:1001 100:1002 200:2001 200:2002 300:3001 400:4001 410:1001 410:6001 500:5001 500:5005 500:5007'
        FROM {TEST_GOLD}
        UNION ALL
        SELECT '04 repository 200, release 2001: the other event is the anchor now', MAX(anchor_summary), 'event=55000000003 r=2024-02-04 09:00:00 tag=nightly pre=false name=org/beta'
        FROM gold_summary WHERE repo_id = 200 AND release_anchor_key = '2001'
        UNION ALL
        SELECT '05 repository 200, release 2001: totals', MAX(count_summary), 'before=0 after=0 change=0 pct=NULL zero=true'
        FROM gold_summary WHERE repo_id = 200 AND release_anchor_key = '2001'
        UNION ALL
        SELECT '06 repository 200, release 2002: totals', MAX(count_summary), 'before=0 after=0 change=0 pct=NULL zero=true'
        FROM gold_summary WHERE repo_id = 200 AND release_anchor_key = '2002'
        UNION ALL
        SELECT '07 repository 200, release 2002: kinds', MAX(kind_summary), 'push=0/0 pr=0/0 issue=0/0 watch=0/0 fork=0/0 eligible=0/0 commits=0/0'
        FROM gold_summary WHERE repo_id = 200 AND release_anchor_key = '2002'
        UNION ALL
        SELECT '08 rows written by run 4', CAST(count_if(_gold_run_id = 'test-run4') AS STRING), '11' FROM {TEST_GOLD}
        UNION ALL
        SELECT '09 audit Silver rows read', MAX(source_summary), 'releases=29 push=12 pull_requests=4 issues=4 watches=6 forks=5'
        FROM audit_summary WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '10 audit reasons', MAX(reason_summary), 'not_published=2 draft=3 no_published_at=1 prerelease_unknown=1 no_release_id=1 repeated=2 not_covered=8'
        FROM audit_summary WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '11 audit quality_metrics', MAX(metrics_summary), 'release_events=29 anchors=11 prerelease=2 baseline_zero=7 without_activity=6 manifest_hours=168 complete_hours=167'
        FROM audit_summary WHERE run_id = 'test-run4'
        UNION ALL
        SELECT '12 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '11'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run4'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T13"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T13: {failed_count} check(s) failed. See the table above.")
print("T13: ALL CHECKS PASSED")

## T14. Run 5: an hour that is not complete, and a source date without rows

Two things are tested in this run:

- **An hour that is not complete.** The hour 2 February 03:00 is marked as not complete in the test manifest. The windows of release 5001 (r 2 Feb 00:00) run from 1 Feb 00:00 to 3 Feb 00:00, so they touch that hour. The other anchors do not.
- **A source date without rows.** The run gets `source_date = 2024-03-01`, a date without any Silver row. A daily Gold table would end as `NO_INPUT`. G5 is rebuilt completely on every run, so it rebuilds the whole table and ends as `SUCCEEDED`.

In [0]:
spark.sql(f"""
    UPDATE {TEST_MANIFEST}
    SET is_complete = false
    WHERE source_hour_utc = TIMESTAMP'2024-02-02 03:00:00'
""")

print("Complete hours in the test manifest:", spark.table(TEST_MANIFEST).filter("is_complete = true").count())

In [0]:
%run ../05_release_activity_shift $test_schema="test_gold_g5" $source_date="2024-03-01" $run_id="test-run5"

## T15. Checks of run 5

- Release 5001 is now `WINDOW_NOT_COVERED`, so it has no row. 10 rows, 9 releases left out by rule 7.
- The manifest has 166 complete hours of 168.
- Every row was written by run 5. A run for one source date still replaces the whole table.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '03 audit source_date', CAST(MAX(source_date) AS STRING), '2024-03-01'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '04 Gold rows', CAST(COUNT(*) AS STRING), '10' FROM {TEST_GOLD}
        UNION ALL
        SELECT '05 the keys of all rows: release 5001 is gone',
               concat_ws(' ', sort_array(collect_list(concat(CAST(repo_id AS STRING), ':', release_anchor_key)))),
               '100:1001 100:1002 200:2001 200:2002 300:3001 400:4001 410:1001 410:6001 500:5005 500:5007'
        FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 rows written by run 5', CAST(count_if(_gold_run_id = 'test-run5') AS STRING), '10' FROM {TEST_GOLD}
        UNION ALL
        SELECT '07 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '10'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '08 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '09 audit error_message', MAX(error_message), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '10 audit reasons', MAX(reason_summary), 'not_published=2 draft=3 no_published_at=1 prerelease_unknown=1 no_release_id=1 repeated=2 not_covered=9'
        FROM audit_summary WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '11 audit quality_metrics', MAX(metrics_summary), 'release_events=29 anchors=10 prerelease=2 baseline_zero=6 without_activity=5 manifest_hours=168 complete_hours=166'
        FROM audit_summary WHERE run_id = 'test-run5'
        UNION ALL
        SELECT '12 audit Silver rows read', MAX(source_summary), 'releases=29 push=12 pull_requests=4 issues=4 watches=6 forks=5'
        FROM audit_summary WHERE run_id = 'test-run5'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T15"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T15: {failed_count} check(s) failed. See the table above.")
print("T15: ALL CHECKS PASSED")

## T16. Run 6: quality gates fail

The Silver notebooks never let such rows into Silver. The test puts two of them into the test tables anyway:

| Bad row | Table | What is wrong | The gate that must catch it |
|---|---|---|---|
| ...32 | Release | A valid release, 7001 at 4 Feb 12:00, but without `repo_id`. It becomes an anchor, and the Gold key would be NULL | `KEY_IS_NULL` |
| ...101 again | Issue | An issue in repository 100 at 4 Feb 13:00 with the `event_id` of the push ...101. One event ID is now in two tables | `EVENT_ID_NOT_UNIQUE` |

The other gates check the arithmetic of the notebook itself, for example that the before total is the sum of its five kinds. No Silver row can make them fail.

The cell below prepares the faults. The run itself is in the cell after it.

In [0]:
spark.sql("INSERT INTO " + TEST_RELEASE + " " + RELEASE_COLUMNS + """
VALUES
    -- A release without repo_id
    ('55000000032', TIMESTAMP'2024-02-04 12:00:05', DATE'2024-02-04', NULL, 'org/broken',       'published', 7001, 'v7.0',      TIMESTAMP'2024-02-04 12:00:00', false, false)
""")

spark.sql("INSERT INTO " + TEST_ISSUE + " " + ACTIVITY_COLUMNS + """
VALUES
    -- The event ID of the push ...101
    ('55000000101', TIMESTAMP'2024-02-04 13:00:00', DATE'2024-02-04', 100, 'org/alpha')
""")

print("Rows in the Silver test tables:")
for table_name in SILVER_TABLE_NAMES:
    print("  ", table_name, ":", spark.table(TEST_SILVER + "." + table_name).count())

### Run 6

**The next cell is EXPECTED to fail** with `RuntimeError: release_activity_shift FAILED: Quality gates failed: KEY_IS_NULL; EVENT_ID_NOT_UNIQUE`. After it has failed, click the cell after it and choose "Run all below".

If this cell runs twice, the audit table gets two rows for run 6. The checks count runs, not rows, so that does no harm.

In [0]:
%run ../05_release_activity_shift $test_schema="test_gold_g5" $source_date="" $run_id="test-run6-fail"

## T17. Checks of run 6: Gold is unchanged

Worked out by hand.

- The run read 30 release events and 5 issues.
- It staged 11 rows: the 10 of run 5 and one row without a repository.
- The staged row without a repository has 0 before and 0 after. The activity is joined on `repo_id`, and in SQL `NULL = NULL` is not true.
- The repeated issue lies in the after window of release 1001 and in the before window of release 1002. So the staged row of 1001 has 4 before and 6 after (change 2, percent change 50.0), and the staged row of 1002 has 5 before and 6 after (change 1, percent change 20.0).
- The two gates are named in the order in which the notebook checks them.
- The run must have written its audit row with the status `FAILED` and the names of the gates.
- The Gold table must be exactly what it was: the 10 rows of run 5. The rejected result is only in the staging table.

After the checks, the test removes the two bad rows, so that the next run can succeed.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'FAILED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '02 audit failed_checks', MAX(failed_checks), 'KEY_IS_NULL; EVENT_ID_NOT_UNIQUE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '03 audit error_message', MAX(error_message), 'Quality gates failed: KEY_IS_NULL; EVENT_ID_NOT_UNIQUE'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '04 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'false'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '05 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '0'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '06 audit Silver rows read', MAX(source_summary), 'releases=30 push=12 pull_requests=4 issues=5 watches=6 forks=5'
        FROM audit_summary WHERE run_id = 'test-run6-fail'
        UNION ALL
        SELECT '07 Gold rows (unchanged)', CAST(COUNT(*) AS STRING), '10' FROM {TEST_GOLD}
        UNION ALL
        SELECT '08 Gold rows still from run 5', CAST(count_if(_gold_run_id = 'test-run5') AS STRING), '10' FROM {TEST_GOLD}
        UNION ALL
        SELECT '09 Gold rows without repo_id', CAST(count_if(repo_id IS NULL) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '10 Gold row of release 1001 (unchanged): totals', MAX(count_summary), 'before=4 after=5 change=1 pct=25.0 zero=false'
        FROM gold_summary WHERE repo_id = 100 AND release_anchor_key = '1001'
        UNION ALL
        SELECT '11 staging rows of the rejected result', CAST(count_if(_gold_run_id = 'test-run6-fail') AS STRING), '11' FROM {TEST_STAGING}
        UNION ALL
        SELECT '12 staging row without repo_id: no activity',
               MAX(concat('release=', release_anchor_key,
                          ' before=', CAST(activity_events_before AS STRING),
                          ' after=', CAST(activity_events_after AS STRING))),
               'release=7001 before=0 after=0'
        FROM {TEST_STAGING} WHERE repo_id IS NULL
        UNION ALL
        SELECT '13 staging row of release 1001: the repeated issue is after',
               MAX(concat('before=', CAST(activity_events_before AS STRING),
                          ' after=', CAST(activity_events_after AS STRING),
                          ' pct=', CAST(activity_events_pct_change AS STRING))),
               'before=4 after=6 pct=50.0'
        FROM {TEST_STAGING} WHERE repo_id = 100 AND release_anchor_key = '1001'
        UNION ALL
        SELECT '14 staging row of release 1002: the repeated issue is before',
               MAX(concat('before=', CAST(activity_events_before AS STRING),
                          ' after=', CAST(activity_events_after AS STRING),
                          ' pct=', CAST(activity_events_pct_change AS STRING))),
               'before=5 after=6 pct=20.0'
        FROM {TEST_STAGING} WHERE repo_id = 100 AND release_anchor_key = '1002'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T17"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T17: {failed_count} check(s) failed. See the table above.")
print("T17: ALL CHECKS PASSED")

In [0]:
spark.sql(f"""
    DELETE FROM {TEST_RELEASE}
    WHERE event_id = '55000000032'
""")

# Only the issue with the repeated ID is removed. The push ...101 stays.
spark.sql(f"""
    DELETE FROM {TEST_ISSUE}
    WHERE event_id = '55000000101'
""")

print("Rows in the Silver test tables after the repair:")
for table_name in SILVER_TABLE_NAMES:
    print("  ", table_name, ":", spark.table(TEST_SILVER + "." + table_name).count())

## T18. Run 7: recovery

The same run again, now without the bad rows. The result must be the one of run 5: 10 rows, now written by run 7.

In [0]:
%run ../05_release_activity_shift $test_schema="test_gold_g5" $source_date="" $run_id="test-run7"

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '02 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '10'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '03 audit checks_passed', CAST(MAX(checks_passed) AS STRING), 'true'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '04 audit failed_checks', MAX(failed_checks), CAST(NULL AS STRING)
        FROM {TEST_AUDIT} WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '05 Gold rows', CAST(COUNT(*) AS STRING), '10' FROM {TEST_GOLD}
        UNION ALL
        SELECT '06 rows written by run 7', CAST(count_if(_gold_run_id = 'test-run7') AS STRING), '10' FROM {TEST_GOLD}
        UNION ALL
        SELECT '07 Gold rows without repo_id', CAST(count_if(repo_id IS NULL) AS STRING), '0' FROM {TEST_GOLD}
        UNION ALL
        SELECT '08 repository 100, release 1001: totals', MAX(count_summary), 'before=4 after=5 change=1 pct=25.0 zero=false'
        FROM gold_summary WHERE repo_id = 100 AND release_anchor_key = '1001'
        UNION ALL
        SELECT '09 repository 100, release 1002: totals', MAX(count_summary), 'before=4 after=6 change=2 pct=50.0 zero=false'
        FROM gold_summary WHERE repo_id = 100 AND release_anchor_key = '1002'
        UNION ALL
        SELECT '10 audit Silver rows read', MAX(source_summary), 'releases=29 push=12 pull_requests=4 issues=4 watches=6 forks=5'
        FROM audit_summary WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '11 audit quality_metrics', MAX(metrics_summary), 'release_events=29 anchors=10 prerelease=2 baseline_zero=6 without_activity=5 manifest_hours=168 complete_hours=166'
        FROM audit_summary WHERE run_id = 'test-run7'
        UNION ALL
        SELECT '12 different runs in the audit table so far', CAST(COUNT(DISTINCT run_id) AS STRING), '8' FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T18"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T18: {failed_count} check(s) failed. See the table above.")
print("T18: ALL CHECKS PASSED")

## T19. Run 8: the hour is complete again

The hour 2 February 03:00 is marked as complete again, as after a reload of its file. A run for all dates must bring release 5001 back.

In [0]:
spark.sql(f"""
    UPDATE {TEST_MANIFEST}
    SET is_complete = true
    WHERE source_hour_utc = TIMESTAMP'2024-02-02 03:00:00'
""")

print("Complete hours in the test manifest:", spark.table(TEST_MANIFEST).filter("is_complete = true").count())

In [0]:
%run ../05_release_activity_shift $test_schema="test_gold_g5" $source_date="" $run_id="test-run8"

## T20. Checks of run 8

Worked out by hand: release 5001 is an anchor again, with 0 before and 0 after. 11 rows, 8 releases left out by rule 7, 167 complete hours. 7 rows with 0 before, 6 of them without any activity.

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 audit status' AS check_name, MAX(status) AS actual, 'SUCCEEDED' AS expected
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '02 audit processing_scope', MAX(processing_scope), 'ALL_DATES'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '03 Gold rows', CAST(COUNT(*) AS STRING), '11' FROM {TEST_GOLD}
        UNION ALL
        SELECT '04 the keys of all rows: release 5001 is back',
               concat_ws(' ', sort_array(collect_list(concat(CAST(repo_id AS STRING), ':', release_anchor_key)))),
               '100:1001 100:1002 200:2001 200:2002 300:3001 400:4001 410:1001 410:6001 500:5001 500:5005 500:5007'
        FROM {TEST_GOLD}
        UNION ALL
        SELECT '05 repository 500, release 5001: anchor', MAX(anchor_summary), 'event=55000000018 r=2024-02-02 00:00:00 tag=r5001 pre=false name=org/zeta'
        FROM gold_summary WHERE repo_id = 500 AND release_anchor_key = '5001'
        UNION ALL
        SELECT '06 repository 500, release 5001: totals', MAX(count_summary), 'before=0 after=0 change=0 pct=NULL zero=true'
        FROM gold_summary WHERE repo_id = 500 AND release_anchor_key = '5001'
        UNION ALL
        SELECT '07 rows written by run 8', CAST(count_if(_gold_run_id = 'test-run8') AS STRING), '11' FROM {TEST_GOLD}
        UNION ALL
        SELECT '08 audit gold_rows_published', CAST(MAX(gold_rows_published) AS STRING), '11'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '09 audit reasons', MAX(reason_summary), 'not_published=2 draft=3 no_published_at=1 prerelease_unknown=1 no_release_id=1 repeated=2 not_covered=8'
        FROM audit_summary WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '10 audit quality_metrics', MAX(metrics_summary), 'release_events=29 anchors=11 prerelease=2 baseline_zero=7 without_activity=6 manifest_hours=168 complete_hours=167'
        FROM audit_summary WHERE run_id = 'test-run8'
        UNION ALL
        SELECT '11 audit rows without an event time', CAST(MAX(rows_excluded['missing_event_timestamp']) AS STRING), '1'
        FROM {TEST_AUDIT} WHERE run_id = 'test-run8'
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T20"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T20: {failed_count} check(s) failed. See the table above.")
print("T20: ALL CHECKS PASSED")

## T21. Bad parameter: `source_date`

`2024-2-3` is not in the form `YYYY-MM-DD`. The notebook must stop in its parameter step, before it reads or writes a table.

**The next cell is EXPECTED to fail** with `ValueError: Invalid source_date '2024-2-3'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../05_release_activity_shift $test_schema="test_gold_g5" $source_date="2024-2-3" $run_id="test-bad-date"

### T21 check: where did it stop?

`%run` runs the other notebook inside this one, so the variables of that notebook can be read here. They show how far the stopped run came:

- `source_date` holds the bad value, so step 1.2 has run.
- `run_id` is still the ID of run 8, so the notebook never came to step 1.4, where it reads the run ID. It stopped before it touched a table.

In [0]:
print("source_date:", source_date)
print("run_id     :", run_id)

if source_date != "2024-2-3":
    raise AssertionError("T21: the run with the bad source_date has not run. Run the cell above first.")

if run_id == "test-bad-date":
    raise AssertionError("T21: the notebook did not stop at the bad source_date. It went on to step 1.4.")

section_results["T21"] = 0
print("T21: the notebook stopped at the bad source_date, as expected")

## T22. Bad parameter: `test_schema`

`TEST_WRONG` has capital letters, so it is not a valid test schema name.

**The next cell is EXPECTED to fail** with `ValueError: Invalid test_schema 'TEST_WRONG'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../05_release_activity_shift $test_schema="TEST_WRONG" $source_date="" $run_id="test-bad-schema"

### T22 check: where did it stop?

- `test_schema` holds the bad value, so step 1.3 has run.
- `run_id` is still the ID of run 8, so the notebook never came to step 1.4.

In [0]:
print("test_schema:", test_schema)
print("run_id     :", run_id)

if test_schema != "TEST_WRONG":
    raise AssertionError("T22: the run with the bad test_schema has not run. Run the cell above first.")

if run_id == "test-bad-schema":
    raise AssertionError("T22: the notebook did not stop at the bad test_schema. It went on to step 1.4.")

section_results["T22"] = 0
print("T22: the notebook stopped at the bad test_schema, as expected")

## T23. Bad parameter: `run_id`

The run ID is written into every Gold row, so it may contain only letters, digits, `-` and `_`. `bad.run.id` contains dots.

**The next cell is EXPECTED to fail** with `ValueError: Invalid run_id 'bad.run.id'`. After it has failed, click the cell after it and choose "Run all below".

In [0]:
%run ../05_release_activity_shift $test_schema="test_gold_g5" $source_date="" $run_id="bad.run.id"

### T23 check: did it run?

- `run_id` holds the bad value, so the notebook came to step 1.4.
- That it stopped there, and wrote nothing, is checked in T24: this run has no audit row and the Gold table is still the one of run 8.

In [0]:
print("test_schema:", test_schema)
print("run_id     :", run_id)

if run_id != "bad.run.id":
    raise AssertionError("T23: the run with the bad run_id has not run. Run the cell above first.")

section_results["T23"] = 0
print("T23: the notebook came to the run_id step. T24 checks that it wrote nothing.")

## T24. Final checks: nothing was written by the three stopped runs, and production is untouched

- The three runs with a bad parameter stopped before the audit step, so the audit table still holds 9 runs and the Gold table is still the one of run 8.
- Every run wrote exactly one audit row. Run 6 is left out of that check, because its cell may have run more than once.
- The versions of the production tables are read again and compared with the versions from T2. The manifest is one of them.

In [0]:
prod_audit_version_after = spark.sql(
    f"DESCRIBE HISTORY {PROD_AUDIT} LIMIT 1"
).collect()[0]["version"]

prod_manifest_version_after = spark.sql(
    f"DESCRIBE HISTORY {PROD_MANIFEST} LIMIT 1"
).collect()[0]["version"]

if spark.catalog.tableExists(PROD_GOLD):
    prod_gold_version_after = spark.sql(
        f"DESCRIBE HISTORY {PROD_GOLD} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_gold_version_after = -1

if spark.catalog.tableExists(PROD_STAGING):
    prod_staging_version_after = spark.sql(
        f"DESCRIBE HISTORY {PROD_STAGING} LIMIT 1"
    ).collect()[0]["version"]
else:
    prod_staging_version_after = -1

print("Production audit version   :", prod_audit_version_after)
print("Production manifest version:", prod_manifest_version_after)
print("Production Gold version    :", prod_gold_version_after)
print("Production staging version :", prod_staging_version_after)

In [0]:
checks = spark.sql(f"""
    SELECT
        check_name,
        actual,
        expected,
        CASE WHEN actual <=> expected THEN 'PASS' ELSE 'FAIL' END AS result
    FROM (
        SELECT '01 different runs in the audit table' AS check_name, CAST(COUNT(DISTINCT run_id) AS STRING) AS actual, '9' AS expected
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '02 audit rows of the three stopped runs', CAST(count_if(run_id IN ('test-bad-date', 'test-bad-schema', 'bad.run.id')) AS STRING), '0'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '03 Gold rows still from run 8', CAST(count_if(_gold_run_id = 'test-run8') AS STRING), '11'
        FROM {TEST_GOLD}
        UNION ALL
        SELECT '04 Gold rows', CAST(COUNT(*) AS STRING), '11'
        FROM {TEST_GOLD}
        UNION ALL
        SELECT '05 runs by status',
               concat('SUCCEEDED=', CAST(COUNT(DISTINCT CASE WHEN status = 'SUCCEEDED' THEN run_id END) AS STRING),
                      ' FAILED=', CAST(COUNT(DISTINCT CASE WHEN status = 'FAILED' THEN run_id END) AS STRING),
                      ' NO_INPUT=', CAST(COUNT(DISTINCT CASE WHEN status = 'NO_INPUT' THEN run_id END) AS STRING)),
               'SUCCEEDED=7 FAILED=1 NO_INPUT=1'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '06 production audit table version unchanged', '{prod_audit_version_after}', '{prod_audit_version_before}'
        UNION ALL
        SELECT '07 production manifest version unchanged', '{prod_manifest_version_after}', '{prod_manifest_version_before}'
        UNION ALL
        SELECT '08 production Gold table version unchanged', '{prod_gold_version_after}', '{prod_gold_version_before}'
        UNION ALL
        SELECT '09 production staging version unchanged', '{prod_staging_version_after}', '{prod_staging_version_before}'
        UNION ALL
        SELECT '10 audit rows of the 8 runs without run 6 (one row each)', CAST(count_if(run_id <> 'test-run6-fail') AS STRING), '8'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '11 audit rows of run 6 are all FAILED', CAST(count_if(run_id = 'test-run6-fail' AND status <> 'FAILED') AS STRING), '0'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '12 audit rows with another pipeline or table name', CAST(count_if(pipeline_name <> 'gold_release_activity_shift' OR gold_table <> '{TEST_GOLD}') AS STRING), '0'
        FROM {TEST_AUDIT}
        UNION ALL
        SELECT '13 audit rows whose scope is not ALL_DATES', CAST(count_if(processing_scope <> 'ALL_DATES') AS STRING), '0'
        FROM {TEST_AUDIT}
    )
    ORDER BY check_name
""")
checks.show(100, truncate=False)

failed_count = checks.filter("result = 'FAIL'").count()
section_results["T24"] = failed_count
if failed_count > 0:
    raise AssertionError(f"T24: {failed_count} check(s) failed. See the table above.")
print("T24: ALL CHECKS PASSED")

## Result of the whole test

Every section wrote its number of failed checks into `section_results`. This cell goes through the 13 sections. A section that is not in the dictionary did not run, for example because the cells above it were skipped.

The test has passed only when this cell prints `TEST PASSED`.

In [0]:
EXPECTED_SECTIONS = [
    "T4", "T7", "T9", "T11", "T13", "T15", "T17",
    "T18", "T20", "T21", "T22", "T23", "T24",
]

problems = []

for section in EXPECTED_SECTIONS:
    if section not in section_results:
        problems.append(section + " did not run")
    elif section_results[section] > 0:
        problems.append(section + " has " + str(section_results[section]) + " failed check(s)")

print("Sections that ran:", len(section_results), "of", len(EXPECTED_SECTIONS))

if len(problems) > 0:
    raise AssertionError("TEST FAILED: " + "; ".join(problems))

print("TEST PASSED: all", len(EXPECTED_SECTIONS), "sections ran and every check passed.")

## Clean up: drop the test schema

Only a schema whose name starts with `test_` is ever dropped.

If the test failed, you can leave this cell out and look at the test tables first. The next test run drops the schema anyway, in T1.

In [0]:
if not TEST_SCHEMA.startswith("test_"):
    raise RuntimeError("Refusing to drop a schema that is not a test schema.")

spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")

print("Dropped:", TEST_SCHEMA)